In [1]:
%load_ext autoreload

%autoreload 2

In [2]:
from typing import Dict, List, Optional, Tuple, Union, Any, Callable, Mapping
import pandas as pd
import pickle
import numpy as np
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, matthews_corrcoef, balanced_accuracy_score, roc_auc_score, cohen_kappa_score, auc, precision_recall_curve)
from rdkit.Chem import rdFingerprintGenerator

In [7]:
with open("data/train_df2.pkl", "rb") as f:
    df1 = pickle.load(f)
with open("data/test_df2.pkl", "rb") as f:
    df2 = pickle.load(f)

df = pd.concat([df1, df2], ignore_index=True)
df

,compounds,inchikey,description,description_lower,description_split_sentence,description_split,description_remove_stop_words,description_phrases,description_gensim,smiles,...,anti_inflammatory_agent,allergen,dye,toxin,flavouring_agent,agrochemical,volatile_oil,antibacterial_agent,insecticide,fp_3_4096
0,"((-)-epicatechin, Oc1cc(O)c2c(c1)O[C@H](c1ccc(...",PFTAWBLQPZVEMU-UKRRQHHQSA-N,"(-)-epicatechin is a catechin with (2R,3R)-con...","(-)-epicatechin is a catechin with (2r,3r)-con...","[(-)-epicatechin is a catechin with (2r,3r)-co...","[[(-)-epicatechin, is, a, catechin, with, (2r,...","[[(-)-epicatechin, catechin, (2r,3r)-configura...","[[(-)-epicatechin, is, a, catechin, with, (2r,...","[(-)-epicatechin, catechin, (2r,3r)-configurat...",Oc1cc(O)c2c(c1)O[C@H](c1ccc(O)c(O)c1)[C@H](O)C2,...,No,No,No,No,No,No,No,No,No,"[188, 231, 266, 315, 361, 416, 467, 589, 656, ..."
1,"(2,6-dichlorobenzonitrile, N#Cc1c(Cl)cccc1Cl, ...",YOYAIZYFCNQIRF-UHFFFAOYSA-N,"2,6-dichlorobenzonitrile is a nitrile that is ...","2,6-dichlorobenzonitrile is a nitrile that is ...","[2,6-dichlorobenzonitrile is a nitrile that is...","[[2,6-dichlorobenzonitrile, is, a, nitrile, th...","[[2,6-dichlorobenzonitrile, nitrile, benzonitr...","[[2,6-dichlorobenzonitrile, is, a, nitrile, th...","[2,6-dichlorobenzonitrile, nitrile, benzonitri...",N#Cc1c(Cl)cccc1Cl,...,No,No,No,No,No,agrochemical,No,No,No,"[213, 358, 561, 974, 1088, 1380, 1683, 1954, 2..."
2,"(aspartame, COC(=O)[C@H](Cc1ccccc1)NC(=O)[C@@H...",IAOZJIPTCAWIRG-QWRGUYRKSA-N,Aspartame is a dipeptide obtained by formal co...,aspartame is a dipeptide obtained by formal co...,[aspartame is a dipeptide obtained by formal c...,"[[aspartame, is, a, dipeptide, obtained, by, f...","[[aspartame, dipeptide, obtained, formal, cond...","[[aspartame, is, a, dipeptide, obtained, by, f...","[aspartame, dipeptide, obtained, formal_conden...",COC(=O)[C@H](Cc1ccccc1)NC(=O)[C@@H](N)CC(=O)O,...,No,No,No,No,flavouring_agent,No,No,No,No,"[32, 54, 70, 86, 117, 277, 389, 420, 509, 695,..."
3,"(azlocillin, CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N...",JTWOMNBEOCYFNV-NFFDBFGFSA-N,Azlocillin is a semisynthetic penicillin havin...,azlocillin is a semisynthetic penicillin havin...,[azlocillin is a semisynthetic penicillin havi...,"[[azlocillin, is, a, semisynthetic, penicillin...","[[azlocillin, semisynthetic, penicillin, havin...","[[azlocillin, is, a, semisynthetic, penicillin...","[azlocillin, semisynthetic, penicillin, having...",CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](NC(=O)N3CCNC3=...,...,No,allergen,No,No,No,No,No,antibacterial_agent,No,"[5, 117, 130, 175, 218, 225, 282, 317, 387, 38..."
4,"(betulinic acid, C=C(C)[C@@H]1CC[C@]2(C(=O)O)C...",QGJZLNKBHJESQX-FZFNOLFKSA-N,Betulinic acid is a pentacyclic triterpenoid t...,betulinic acid is a pentacyclic triterpenoid t...,[betulinic acid is a pentacyclic triterpenoid ...,"[[betulinic, acid, is, a, pentacyclic, triterp...","[[betulinic_acid, pentacyclic, triterpenoid, l...","[[betulinic_acid, is, a, pentacyclic, triterpe...","[betulinic_acid, pentacyclic, triterpenoid, lu...",C=C(C)[C@@H]1CC[C@]2(C(=O)O)CC[C@]3(C)[C@H](CC...,...,anti_inflammatory_agent,No,No,No,No,No,No,No,No,"[89, 138, 202, 267, 310, 478, 519, 527, 549, 5..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3921,"(cefpirome, CO/N=C(\C(=O)N[C@@H]1C(=O)N2C(C(=O...",DKOQGJHPHLTOJR-WHRDSVKCSA-N,Cefpirome is a fourth-generation cephalosporin...,cefpirome is a fourth-generation cephalosporin...,[cefpirome is a fourth-generation cephalospori...,"[[cefpirome, is, a, fourth-generation, cephalo...","[[cefpirome, fourth-generation, cephalosporin,...","[[cefpirome, is, a, fourth-generation, cephalo...","[cefpirome, fourth-generation, cephalosporin, ...",CO/N=C(\C(=O)N[C@@H]1C(=O)N2C(C(=O)[O-])=C(C[n...,...,No,allergen,No,No,No,No,No,No,No,"[5, 104, 109, 150, 243, 316, 378, 387, 500, 54..."
3922,"(corilagin, O=C(O[C@@H]1O[C@@H]2COC(=O)c3cc(O)...",TUSDEZXZIZRFGC-XIGLUPEJSA-N,Corilagin is an ellagitannin with a

In [8]:
with open("data/10genre_all.pkl", "wb") as f:
    pickle.dump(df, f)

In [5]:
def fin(df, radius, fpSize):
    fingerprints = []
    onbits_list = []
    fp_generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fpSize)
    for i, mol in enumerate(df["ROMol"]):
        try:
            fp = fp_generator.GetFingerprint(mol)
            # 1になっているビットの位置を取得
            onbits = list(fp.GetOnBits())
            onbits_list.append(onbits)
            
            # NumPy配列も必要なら
            fp_np = fp_generator.GetFingerprintAsNumPy(mol)
            fingerprints.append(fp_np)

        except Exception as e:
            print(f"Error processing molecule {i}: {e}")
            continue
    return np.array(fingerprints), onbits_list

def add_vectors(fp_list: List[List[int]], model: Doc2Vec) -> List[np.ndarray]:
    """Combine document vectors based on fingerprints
    
    Args:
        fp_list: List of fingerprint lists, where each fingerprint is represented as a list of indices
        model: Trained Doc2Vec model containing document vectors
        
    Returns:
        List of compound vectors as numpy arrays
    """
    compound_vec = []
    for i in fp_list:
        fingerprint_vec = 0
        for j in i:
            fingerprint_vec += model.dv.vectors[j]
        compound_vec.append(fingerprint_vec)
    return compound_vec

def calculate_metrics(y_true, y_pred, y_proba):

    metrics = {}
    metrics['f1'] = f1_score(y_true, y_pred)
    metrics['mcc'] = matthews_corrcoef(y_true, y_pred)
    metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)
    metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    metrics['kappa'] = cohen_kappa_score(y_true, y_pred)
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    metrics['pr_auc'] = auc(recall, precision)
    return metrics

def evaluate_category(category: str, 
                      X_vec: np.ndarray, 
                      y: np.ndarray, 
                      lightgbm_model
                      ) -> Dict[str, Union[List[float], float]]:
    
    # 全ての評価指標のスコアを格納する辞書
    all_train_scores = {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}
    all_test_scores = {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}


    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    for train_idx, test_idx in skf.split(range(len(y)), y):
        X_train_vec, X_test_vec = X_vec[train_idx], X_vec[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        lightgbm_model.fit(X_train_vec, y_train)
        y_train_pred = lightgbm_model.predict(X_train_vec)
        y_test_pred = lightgbm_model.predict(X_test_vec)
        y_train_proba = lightgbm_model.predict_proba(X_train_vec)[:, 1]
        y_test_proba = lightgbm_model.predict_proba(X_test_vec)[:, 1]

        train_metrics = calculate_metrics(y_train, y_train_pred, y_train_proba)
        test_metrics = calculate_metrics(y_test, y_test_pred, y_test_proba)
        for metric_name in all_train_scores.keys():
            all_train_scores[metric_name].append(train_metrics[metric_name])
            all_test_scores[metric_name].append(test_metrics[metric_name])
    
    # 結果を整理
    results = {}
    for metric_name in all_train_scores.keys():
        results[metric_name] = {
            'train_scores': all_train_scores[metric_name],
            'test_scores': all_test_scores[metric_name],
            'mean_train': np.mean(all_train_scores[metric_name]),
            'mean_test': np.mean(all_test_scores[metric_name])
        }
    
    return results

def build_fpdoc2vec_model(purpose_description, 
                        tag_list, #変更
                        doc2vec_param) :
    """
    Build and train a Doc2Vec model from corpus and structure information
    
    Args:
        corpus: List of lists containing tokenized text for each document
        list: List of lists containing tags for each document
        doc2vec_param: Dictionary of parameters for the Doc2Vec model
        
    Returns:
        Trained Doc2Vec model
    """
    corpus = df[purpose_description].tolist()
    tagged_documents = [
        TaggedDocument(words=corpus, tags=tag_list[i]) #変更
        for i, corpus in enumerate(corpus)
    ]
    
    model = Doc2Vec(tagged_documents, **doc2vec_param)
    
    return model

def make_doc2vector(model, tag_list) -> np.ndarray:

    compound_vec = add_vectors(tag_list, model)
    X_vec = np.array(compound_vec)
    return X_vec

def main(df: pd.DataFrame, 
         X_vec: np.ndarray, 
         lightgbm_model) -> Dict[str, Dict[str, float]]:
    """
    Main function to train and evaluate compound classification models using provided features and Doc2Vec.
    
    Args:
        input_path: Path to the pickle file containing compound data
        feature_list: List of molecular features (like fingerprints) to use in the model
        doc2vec_param: Parameters for the Doc2Vec model
        lightgbm_model: Pre-configured LightGBM classifier
        purpose_description: Column name in the DataFrame containing text descriptions
        
    Returns:
        Dictionary mapping category names to evaluation results
    """
        
    # Define categories to evaluate
    categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
    
    # Evaluate each category
    results = {}
    for category in categories:
        y = np.array([1 if i == category else 0 for i in df[category]])
        results[category] = evaluate_category(category, X_vec, y, lightgbm_model)

    return results

!!!! Fp doc2vec !!!

In [17]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

doc2vec_param: Dict[str, Any] = {
    'vector_size': 150,
    'dm': 1,
    'window': 9,
    'min_count': 0,
    'alpha': 0.014996471116783728,
    'sample': 4.4713728630733355e-05,
    'epochs': 840,
    'negative': 12,
    'workers': 1,
    'seed': 0
}

gbm_params: Dict[str, Any] = {
    "boosting_type": "dart",
    "num_leaves": 48,
    "max_depth": 5,
    "learning_rate": 0.04166324251391809,
    "n_estimators": 736,
    "class_weight": "balanced",
    "min_split_gain": 0.009346925180781129,
    "min_child_weight": 0.0007929549087822909,
    "min_child_samples": 37,
    "reg_alpha": 1.757104268180148,
    "reg_lambda": 1.463369722508726,
    "feature_fraction": 0.50163362868711,
    "feature_fraction_bynode": 0.8321043377994284,
    "subsample": 0.6974385909748512,
    "colsample_bytree": 0.6568268046410831,
    "subsample_freq": 5,
    "drop_rate": 0.24668126335938073,
    "max_drop": 28,
    "skip_drop": 0.5591506516119614,
    "uniform_drop": True,
    "xgboost_dart_mode": True,
    "objective": "binary",
    "random_state": 0,
    "verbose": -1,
    "force_col_wise": True
}

# Create classifier
lightgbm_model = lgb.LGBMClassifier(**gbm_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    df = pickle.load(f)

# FP Doc2Vec approach

fp4096_list, bit_list = fin(df, 3, 8192)
model = build_fpdoc2vec_model("description_gensim", bit_list, doc2vec_param)
model.save("train_df_doc2vec_8192.model")
X_vec = make_doc2vector(model, bit_list)
results8192 = main(df, X_vec, lightgbm_model)

In [19]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in results8192.items():
    print(f"## {category} ##")
    print(results8192[category]['f1']["mean_test"])
    li.append(results8192[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.7293207359089713
## anti_inflammatory_agent ##
0.7410584426683087
## allergen ##
0.663756864405479
## dye ##
0.9389962296700267
## toxin ##
0.599149768169376
## flavouring_agent ##
0.7119200986806621
## agrochemical ##
0.817615114799087
## volatile_oil ##
0.8123782073057889
## antibacterial_agent ##
0.7004140218637552
## insecticide ##
0.7658216985009149

0.7480431181972371


In [20]:
with open("result_full_prediction/fpdoc2vec.pkl", "rb") as f:
    a = pickle.load(f)
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in a.items():
    print(f"## {category} ##")
    print(a[category]['f1']["mean_test"])
    li.append(a[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.7221319778859124
## anti_inflammatory_agent ##
0.7467746235518791
## allergen ##
0.6912898979130322
## dye ##
0.9385639376334233
## toxin ##
0.6287459557985874
## flavouring_agent ##
0.7306613370362036
## agrochemical ##
0.8155121280984631
## volatile_oil ##
0.8023856634836968
## antibacterial_agent ##
0.6956427669841374
## insecticide ##
0.7729619861371464

0.7544670274522483


In [52]:
with open("result_full_prediction/fpdoc2vec.pkl", "wb") as f:
    pickle.dump(results4096, f)

!!! NAME Doc2vec !!!

In [6]:
def make_name2vector(df: pd.DataFrame, purpose_description, doc2vec_param) -> np.ndarray:
    """Convert to compound vectors using NameDoc2Vec model
    
    Args:
        model_path: Path to the saved NameDoc2Vec model file
        df: DataFrame containing compound data
        
    Returns:
        NumPy array of document vectors with shape (len(df), vector_size)
    """
    corpus = df[purpose_description].tolist()#変更
    tag_list = [i[0] for i in df["compounds"]]

    tagged_documents = [TaggedDocument(words=corpus, tags=[tag_list[i]]) #変更
        for i, corpus in enumerate(corpus)]
    model = Doc2Vec(tagged_documents, **doc2vec_param)
    X_vec = np.array([model.dv.vectors[i] for i in range(len(df))])

    return X_vec

In [7]:
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    df = pickle.load(f)
X_vec = make_name2vector(df, "description_gensim", doc2vec_param)
namedoc_results = main(df, X_vec, lightgbm_model)

In [15]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in namedoc_results.items():
    print(f"## {category} ##")
    print(namedoc_results[category]['mcc']["mean_test"])
    li.append(namedoc_results[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.7378533648897728
## anti_inflammatory_agent ##
0.7186814062419942
## allergen ##
0.7987338455124797
## dye ##
0.9063205701082053
## toxin ##
0.6762032529866542
## flavouring_agent ##
0.7760785112966042
## agrochemical ##
0.765286888865379
## volatile_oil ##
0.8481418357168101
## antibacterial_agent ##
0.7272062582684062
## insecticide ##
0.7636352069632788

0.7718141140849586


In [68]:
with open("result_full_prediction/namedoc2vec.pkl", "wb") as f:
    pickle.dump(namedoc_results, f)

In [6]:
input_path = "result_full_prediction/namedoc2vec.pkl"
with open(input_path, "rb") as f:
    a = pickle.load(f)
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in a.items():
    print(f"## {category} ##")
    print(a[category]['f1']["mean_test"])
    li.append(a[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.7671965214779315
## anti_inflammatory_agent ##
0.7660120243499368
## allergen ##
0.8154858581540173
## dye ##
0.9211018323633697
## toxin ##
0.6794094990208336
## flavouring_agent ##
0.7827436383626024
## agrochemical ##
0.7903278754183274
## volatile_oil ##
0.855548446133812
## antibacterial_agent ##
0.7709361024248991
## insecticide ##
0.7786183354561267

0.7927380133161857


!!! ECFP 4096bit !!!

In [9]:
ecfp, bit_list = fin(df, 3, 4096)
ecfp_results = main(df, ecfp, lightgbm_model)

In [16]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in ecfp_results.items():
    print(f"## {category} ##")
    print(ecfp_results[category]['mcc']["mean_test"])
    li.append(ecfp_results[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.545193113699051
## anti_inflammatory_agent ##
0.5287862713537379
## allergen ##
0.5086127559424773
## dye ##
0.8643152415468449
## toxin ##
0.3982942714113883
## flavouring_agent ##
0.5336110369749492
## agrochemical ##
0.6219490805760459
## volatile_oil ##
0.6762211491101774
## antibacterial_agent ##
0.4728389395302502
## insecticide ##
0.5526935320545044

0.5702515392199425


In [60]:
with open("result_full_prediction/ecfp.pkl", "wb") as f:
    pickle.dump(ecfp_results, f)

!!! Descriptor !!!

In [3]:
def make_descriptors(
    input_file: str, 
    train_df: pd.DataFrame, 
) -> Dict[str, Dict[str, float]]:

    # Load dataset
    with open(input_file, "rb") as f:
        df = pickle.load(f)
        
    # Split data into train and test sets
    train_df1 = df[df["inchikey"].isin(list(train_df["inchikey"]))]
    test_df1 = df.drop(train_df1.index)
    
    # Extract descriptor columns (from column 14 onward)
    train_desc = np.array(train_df1.select_dtypes(include=[np.floating]))
    test_desc = np.array(test_df1.select_dtypes(include=[np.floating]))

    return train_desc, test_desc

In [8]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

with open("data/train_df2.pkl", "rb") as f:
    df = pickle.load(f)
gbm_params: Dict[str, Any] = {
    "boosting_type": "dart",
    "num_leaves": 48,
    "max_depth": 5,
    "learning_rate": 0.04166324251391809,
    "n_estimators": 736,
    "class_weight": "balanced",
    "min_split_gain": 0.009346925180781129,
    "min_child_weight": 0.0007929549087822909,
    "min_child_samples": 37,
    "reg_alpha": 1.757104268180148,
    "reg_lambda": 1.463369722508726,
    "feature_fraction": 0.50163362868711,
    "feature_fraction_bynode": 0.8321043377994284,
    "subsample": 0.6974385909748512,
    "colsample_bytree": 0.6568268046410831,
    "subsample_freq": 5,
    "drop_rate": 0.24668126335938073,
    "max_drop": 28,
    "skip_drop": 0.5591506516119614,
    "uniform_drop": True,
    "xgboost_dart_mode": True,
    "objective": "binary",
    "random_state": 0,
    "verbose": -1,
    "force_col_wise": True
}

# Create classifier
lightgbm_model = lgb.LGBMClassifier(**gbm_params)
input_descriptor_path = "data/10genre_32descriptor2.pkl"
desc = make_descriptors(input_descriptor_path, df)[0]
desc_results = main(df, desc, lightgbm_model)

In [9]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in desc_results.items():
    print(f"## {category} ##")
    print(desc_results[category]['f1']["mean_test"])
    li.append(desc_results[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6166555384400485
## anti_inflammatory_agent ##
0.6115340673598899
## allergen ##
0.6354464072212129
## dye ##
0.8720457470640192
## toxin ##
0.4913845128244462
## flavouring_agent ##
0.6761305625741247
## agrochemical ##
0.7350694110371245
## volatile_oil ##
0.7893113320193262
## antibacterial_agent ##
0.5745773959696975
## insecticide ##
0.6986054140727423

0.6700760388582632


In [32]:
with open("result_full_prediction/descriptor2.pkl", "wb") as f:
    pickle.dump(desc_results, f)

In [4]:
from sklearn.linear_model import LogisticRegression

！！！Logistic Regeression

!!! FP doc2vec

In [84]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

doc2vec_param: Dict[str, Any] = {
    'vector_size': 150,
    'dm': 1,
    'window': 9,
    'min_count': 0,
    'alpha': 0.014996471116783728,
    'sample': 4.4713728630733355e-05,
    'epochs': 840,
    'negative': 12,
    'workers': 1,
    'seed': 0
}

lr_params: Dict[str, Any] = {
       "C": 5.378351934170314, 
       "penalty": "l1", 
       "max_iter": 4300, 
       "class_weight": None, 
       "tol": 0.003446628082848086, 
       "solver": "liblinear", 
       "random_state": 0
}


# Create classifier
lr = LogisticRegression(**lr_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    df = pickle.load(f)

# FP Doc2Vec approach
fp4096_list, bit_list = fin(df, 3, 4096)
X_vec = make_doc2vector(df, "description_gensim", bit_list, doc2vec_param)
results4096 = main(df, X_vec, lr)

In [85]:
li = []
for category, result in results4096.items():
    print(f"## {category} ##")
    print(results4096[category]['mcc']["mean_test"])
    li.append(results4096[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.696220988781943
## anti_inflammatory_agent ##
0.7237638338455377
## allergen ##
0.7695875787238953
## dye ##
0.8954979228186806
## toxin ##
0.5356778110653008
## flavouring_agent ##
0.7195590841809821
## agrochemical ##
0.7972611604932802
## volatile_oil ##
0.7877434169624531
## antibacterial_agent ##
0.6990244416442334
## insecticide ##
0.7765469157592506

0.7400883154275557


In [41]:
with open("result_full_prediction/fpdoc2vec_lr.pkl", "wb") as f:
    pickle.dump(results4096, f)

!!! name doc2vec

In [86]:
lr_params: Dict[str, Any] = {
       "C": 5.378351934170314, 
       "penalty": "l1", 
       "max_iter": 4300, 
       "class_weight": None, 
       "tol": 0.003446628082848086, 
       "solver": "liblinear", 
       "random_state": 0
}

# Create classifier
lr = LogisticRegression(**lr_params)

input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    df = pickle.load(f)
X_name_vec = make_name2vector(df, "description_gensim", doc2vec_param)
namedoc_results = main(df, X_name_vec, lr)

In [92]:
li = []
for category, result in namedoc_results.items():
    print(f"## {category} ##")
    print(namedoc_results[category]['f1']["mean_test"])
    li.append(namedoc_results[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.7580203228298741
## anti_inflammatory_agent ##
0.7251145466706164
## allergen ##
0.8063261262813335
## dye ##
0.8755309433741404
## toxin ##
0.5936788942052099
## flavouring_agent ##
0.7949034725449258
## agrochemical ##
0.6946167603459622
## volatile_oil ##
0.7904782027589045
## antibacterial_agent ##
0.7791578116276854
## insecticide ##
0.7346699502886116

0.7552497030927264


In [14]:
gbm_params: Dict[str, Any] = {
    "boosting_type": "dart",
    "num_leaves": 48,
    "max_depth": 5,
    "learning_rate": 0.04166324251391809,
    "n_estimators": 736,
    "class_weight": "balanced",
    "min_split_gain": 0.009346925180781129,
    "min_child_weight": 0.0007929549087822909,
    "min_child_samples": 37,
    "reg_alpha": 1.757104268180148,
    "reg_lambda": 1.463369722508726,
    "feature_fraction": 0.50163362868711,
    "feature_fraction_bynode": 0.8321043377994284,
    "subsample": 0.6974385909748512,
    "colsample_bytree": 0.6568268046410831,
    "subsample_freq": 5,
    "drop_rate": 0.24668126335938073,
    "max_drop": 28,
    "skip_drop": 0.5591506516119614,
    "uniform_drop": True,
    "xgboost_dart_mode": True,
    "objective": "binary",
    "random_state": 0,
    "verbose": -1,
    "force_col_wise": True
}

# Create classifier
lightgbm_model = lgb.LGBMClassifier(**gbm_params)

# Define categories to evaluate
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
    
# Evaluate each category
results = {}
for category in categories:
    print(f"##{category}##")
    y = np.array([1 if i == category else 0 for i in df[category]])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    li = []
    for train_idx, test_idx in skf.split(range(len(y)), y):
        X_train_vec, X_test_vec = X_vec[train_idx], X_vec[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        lightgbm_model.fit(X_train_vec, y_train)
        y_train_pred = lightgbm_model.predict(X_train_vec)
        y_test_pred = lightgbm_model.predict(X_test_vec)
        test_score = f1_score(y_test, y_test_pred)
        train_score = f1_score(y_train, y_train_pred)   

        li.append((train_score, test_score))
    train_score = np.mean([x[0] for x in li])
    test_score = np.mean([x[1] for x in li])
    print(f"Train scores: {train_score}")
    print(f"Test scores: {test_score}")

##antioxidant##
Train scores: 0.999734395750332
Test scores: 0.7671965214779315
##anti_inflammatory_agent##
Train scores: 0.9968558004479228
Test scores: 0.7660120243499368
##allergen##
Train scores: 0.999644760213144
Test scores: 0.8154858581540173
##dye##
Train scores: 0.999791013584117
Test scores: 0.9211018323633697
##toxin##
Train scores: 1.0
Test scores: 0.6794094990208336
##flavouring_agent##
Train scores: 0.99736406427528
Test scores: 0.7827436383626024
##agrochemical##
Train scores: 0.9957575153766122
Test scores: 0.7903278754183274
##volatile_oil##
Train scores: 1.0
Test scores: 0.855548446133812
##antibacterial_agent##
Train scores: 0.9984286759879446
Test scores: 0.7709361024248991
##insecticide##
Train scores: 0.9963668783238834
Test scores: 0.7786183354561267


In [46]:
with open("result_full_prediction/namedoc2vec_lr.pkl", "wb") as f:
    pickle.dump(namedoc_results, f)

!!! ECFP

In [88]:
ecfp, bit_list = fin(df, 3, 4096)
ecfp_results = main(df, ecfp, lr)

In [93]:
li = []
for category, result in ecfp_results.items():
    print(f"## {category} ##")
    print(ecfp_results[category]['f1']["mean_test"])
    li.append(ecfp_results[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6424682818862362
## anti_inflammatory_agent ##
0.6475409046016257
## allergen ##
0.6130605074536958
## dye ##
0.9067637845805481
## toxin ##
0.5058786216469631
## flavouring_agent ##
0.581521012960563
## agrochemical ##
0.7241784266595406
## volatile_oil ##
0.7798791244950427
## antibacterial_agent ##
0.5616115083363992
## insecticide ##
0.7130454127281702

0.6675947585348784


In [50]:
with open("result_full_prediction/ecfp_lr.pkl", "wb") as f:
    pickle.dump(ecfp_results, f)

!!! Descriptors

In [90]:
input_descriptor_path = "data/10genre_32descriptor.pkl"
desc = make_descriptors(input_descriptor_path, df)[0]
desc_results = main(df, desc, lr)

In [91]:
li = []
for category, result in desc_results.items():
    print(f"## {category} ##")
    print(desc_results[category]['f1']["mean_test"])
    li.append(desc_results[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.377085323441175
## anti_inflammatory_agent ##
0.24588655527846393
## allergen ##
0.21314895154858893
## dye ##
0.803673664140472
## toxin ##
0.03678646934460888
## flavouring_agent ##
0.29342515542802483
## agrochemical ##
0.46244678992044574
## volatile_oil ##
0.7295480511164365
## antibacterial_agent ##
0.1767187040284414
## insecticide ##
0.29935525578095856

0.3638074920027616


In [69]:
with open("result_full_prediction/descriptor_lr.pkl", "wb") as f:
    pickle.dump(desc_results, f)

In [25]:
with open("result_full_prediction/___delendpoint_fpdoc2vec.pkl", "rb") as f:
    a = pickle.load(f)

In [28]:
li = []
for category, result in a.items():
    print(f"## {category} ##")
    print(a[category]['f1']["mean_test"])
    li.append(a[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.7149774903022681
## anti_inflammatory_agent ##
0.7346487217238961
## allergen ##
0.6630551869357839
## dye ##
0.935288703746688
## toxin ##
0.6264190753110063
## flavouring_agent ##
0.7045410357157346
## agrochemical ##
0.8015671585396145
## volatile_oil ##
0.8089078895988694
## antibacterial_agent ##
0.6716627755038391
## insecticide ##
0.7442267448545212

0.7405294782232221


！！！Toxin特化モデル！！！

!!!Fpdoc2vec!!!

In [8]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

doc2vec_param: Dict[str, Any] = {
    'vector_size': 150,
    'dm': 1,
    'window': 5,
    'min_count': 0,
    'alpha': 0.048025794302672266,
    'sample': 8.267052687937774e-05,
    'epochs': 410,
    'negative': 14,
    'workers': 1,
    'seed': 0
}

gbm_params: Dict[str, Any] = {
    "boosting_type": "dart",
    "num_leaves": 232,
    "max_depth": 8,
    "learning_rate": 0.13148212280788416,
    "n_estimators": 724,
    "class_weight": "balanced",
    "min_split_gain": 0.08625194938095579,
    "min_child_weight": 0.0001119406781826523,
    "min_child_samples": 49,
    "reg_alpha": 1.45299930462245,
    "reg_lambda": 1.3108801016294607,
    "feature_fraction": 0.7161619999674242,
    "feature_fraction_bynode": 0.7003849648836373,
    "subsample": 0.5596715055348624,
    "colsample_bytree": 0.5068084541058617,
    "subsample_freq": 9,
    "drop_rate": 0.11197887287190003,
    "max_drop": 40,
    "skip_drop": 0.4197667702758505,
    "uniform_drop": False,
    "xgboost_dart_mode": True,
    "objective": "binary",
    "random_state": 0,
    "verbose": -1,
    "force_col_wise": True
}

# Create classifier
lightgbm_model = lgb.LGBMClassifier(**gbm_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    df = pickle.load(f)

# FP Doc2Vec approach
fp4096_list, bit_list = fin(df, 3, 4096)
X_vec = make_doc2vector(df, "description_gensim", bit_list, doc2vec_param)
toxin_results4096 = main(df, X_vec, lightgbm_model)

In [9]:
li = []
for category, result in toxin_results4096.items():
    print(f"## {category} ##")
    print(toxin_results4096[category]['f1']["mean_test"])
    li.append(toxin_results4096[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.7593455454153114
## anti_inflammatory_agent ##
0.7747782750931893
## allergen ##
0.7286048028029233
## dye ##
0.9453238991413186
## toxin ##
0.6410094429058433
## flavouring_agent ##
0.729603149988121
## agrochemical ##
0.844016233425652
## volatile_oil ##
0.8234276906701139
## antibacterial_agent ##
0.7329345384882295
## insecticide ##
0.7992765438081217

0.7778320121738824


In [10]:
with open("result_full_prediction/TOtoxin_fpdoc2vec.pkl", "wb") as f:
    pickle.dump(toxin_results4096, f)

!!!Namedoc2vec!!!

In [12]:
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    df = pickle.load(f)
X_vec = make_name2vector(df, "description_gensim", doc2vec_param)
namedoc_results = main(df, X_vec, lightgbm_model)

In [13]:
li = []
for category, result in namedoc_results.items():
    print(f"## {category} ##")
    print(namedoc_results[category]['f1']["mean_test"])
    li.append(namedoc_results[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.7836458628912981
## anti_inflammatory_agent ##
0.7722577778910594
## allergen ##
0.8219354798466979
## dye ##
0.89062252264872
## toxin ##
0.6359911720513777
## flavouring_agent ##
0.774951489758949
## agrochemical ##
0.7753795709734179
## volatile_oil ##
0.8323835826620638
## antibacterial_agent ##
0.7704268096608409
## insecticide ##
0.7846466936212853

0.784224096200571


In [14]:
with open("result_full_prediction/TOtoxin_namedoc2vec.pkl", "wb") as f:
    pickle.dump(namedoc_results, f)

!!!ECFP!!!

In [15]:
ecfp, bit_list = fin(df, 3, 4096)
toxinecfp_results = main(df, ecfp, lightgbm_model)

In [16]:
li = []
for category, result in toxinecfp_results.items():
    print(f"## {category} ##")
    print(toxinecfp_results[category]['f1']["mean_test"])
    li.append(toxinecfp_results[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6132976218792319
## anti_inflammatory_agent ##
0.6361598369381054
## allergen ##
0.5570183713800322
## dye ##
0.8934746819471624
## toxin ##
0.43432736356588536
## flavouring_agent ##
0.5430630903421159
## agrochemical ##
0.6795214593407664
## volatile_oil ##
0.6843306225759616
## antibacterial_agent ##
0.5633162033188694
## insecticide ##
0.5998847480543779

0.6204393999342508


In [17]:
with open("result_full_prediction/TOtoxin_ecfp.pkl", "wb") as f:
    pickle.dump(toxinecfp_results, f)

Descriptors

In [20]:
input_descriptor_path = "data/10genre_32descriptor.pkl"
desc = make_descriptors(input_descriptor_path, df)[0]
desc_results = main(df, desc, lightgbm_model)

In [21]:
li = []
for category, result in desc_results.items():
    print(f"## {category} ##")
    print(desc_results[category]['f1']["mean_test"])
    li.append(desc_results[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6315753243192335
## anti_inflammatory_agent ##
0.6094135633148315
## allergen ##
0.6368495965499152
## dye ##
0.8838348995446118
## toxin ##
0.5198026832580476
## flavouring_agent ##
0.6639979608321428
## agrochemical ##
0.7296456617197655
## volatile_oil ##
0.7926398413616458
## antibacterial_agent ##
0.574403469048234
## insecticide ##
0.680162429378531

0.672232542932696


In [22]:
with open("result_full_prediction/TOtoxin_descriptor.pkl", "wb") as f:
    pickle.dump(desc_results, f)